# Fine-tune TrOCR for Arabic ID Card OCR

Trains an Arabic OCR model that is robust to blur and poor camera quality.

**Strategy**: Generate synthetic Arabic text images (exact labels, no noise) and apply heavy blur/noise augmentation. The model learns to read Arabic letters reliably under bad conditions — independent of any specific ID card.

**Runtime**: T4 GPU, ~1-2 hours for 10k images / 20 epochs.

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q transformers datasets Pillow torch torchvision tqdm editdistance arabic-reshaper python-bidi
!apt-get install -q fonts-amiri fonts-arabeyes   # Arabic fonts for rendering

In [ ]:
# ── 2. Mount Drive ─────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/trocr-arabic-id'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── 3. Arabic vocabulary (covers all ID card field types) ──────────────────
import random

ARABIC_NAMES = [
    "محمد", "أحمد", "علي", "عمر", "إبراهيم", "خالد", "يوسف", "عبدالله",
    "مصطفى", "حسن", "حسين", "طارق", "كريم", "سامي", "وليد", "هاني",
    "فاطمة", "مريم", "نور", "سارة", "هند", "دينا", "رانيا", "أميرة",
    "ياسمين", "منى", "سمر", "غادة", "نادية", "هالة", "إيمان", "شيماء",
    "عبدالرحمن", "عبدالعزيز", "شحاتة", "سيد", "البدوي", "الشيخ",
    "الحسيني", "الأنصاري", "الشافعي", "الزيات", "محمود", "رمضان",
]

ARABIC_ADDRESSES = [
    "شارع التحرير", "ميدان رمسيس", "شارع النيل", "شارع السلام",
    "شارع العروبة", "ميدان الجيزة", "شارع البحر", "حي العجوزة",
    "شارع الجمهورية", "شارع الهرم", "حي الدقي", "حي المعادي",
    "الاسكندرية", "القاهرة", "الجيزة", "الإسماعيلية", "السويس",
    "المنصورة", "طنطا", "أسيوط", "المنيا", "سوهاج", "أسوان",
    "شارع الهادي", "حي المنتزه", "عصافرة", "سيدي بشر",
]

MISC_WORDS = [
    "مهندس", "طبيب", "معلم", "موظف", "تاجر", "محامٍ", "محاسب",
    "ذكر", "أنثى", "مسلم", "مسيحي", "مسلمة", "مسيحية",
    "أعزب", "متزوج", "متزوجة", "أرمل", "أرملة",
    "ربة منزل", "طالب", "أستاذ", "مدير", "فني",
]

ARABIC_INDIC = "٠١٢٣٤٥٦٧٨٩"

def rand_num(n): return "".join(random.choice(ARABIC_INDIC) for _ in range(n))
def rand_date(): return f"{rand_num(2)}/{rand_num(2)}/١٩{rand_num(2)}"
def rand_id():   return rand_num(14)

def sample_text():
    cat = random.choices(
        ["name", "address", "misc", "id", "date", "serial"],
        weights=[30, 20, 15, 15, 15, 5]
    )[0]
    if cat == "name":
        return " ".join(random.choice(ARABIC_NAMES) for _ in range(random.randint(1,3)))
    elif cat == "address":
        parts = random.sample(ARABIC_ADDRESSES, random.randint(1,2))
        prefix = rand_num(random.randint(1,2)) + " " if random.random() < 0.4 else ""
        return prefix + " ".join(parts)
    elif cat == "misc":
        return random.choice(MISC_WORDS)
    elif cat == "id":
        return rand_id()
    elif cat == "date":
        return rand_date()
    else:
        return rand_num(random.randint(7, 9))

# Preview
for _ in range(6): print(sample_text())

In [ ]:
# ── 4. Synthetic image renderer ────────────────────────────────────────────
import os, glob
import arabic_reshaper
from bidi.algorithm import get_display
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import numpy as np

# Find installed Arabic fonts
FONT_PATHS = glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
FONT_PATHS = [f for f in FONT_PATHS if any(k in f.lower() for k in ['amiri','arab','noto'])]
if not FONT_PATHS:
    FONT_PATHS = glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)[:5]
print(f"Fonts found: {[os.path.basename(f) for f in FONT_PATHS]}")

_font_cache = {}
def get_font(path, size):
    if (path, size) not in _font_cache:
        _font_cache[(path, size)] = ImageFont.truetype(path, size)
    return _font_cache[(path, size)]

def reshape(text):
    """Reshape + reorder Arabic so PIL renders properly connected letters."""
    return get_display(arabic_reshaper.reshape(text))

def make_bg(w, h):
    style = random.choice(['white', 'offwhite', 'tinted'])
    if style == 'white':
        return Image.new('RGB', (w, h), (255, 255, 255))
    elif style == 'offwhite':
        v = random.randint(220, 255)
        return Image.new('RGB', (w, h), (v, v, v))
    else:
        r,g,b = random.randint(200,255), random.randint(200,255), random.randint(200,255)
        return Image.new('RGB', (w, h), (r, g, b))

def render(text, font_path, h=64, pad=10):
    display = reshape(text)
    size = int(h * 0.65)
    font = get_font(font_path, size)
    dummy = ImageDraw.Draw(Image.new('RGB', (1,1)))
    bb = dummy.textbbox((0,0), display, font=font)
    tw, th = bb[2]-bb[0], bb[3]-bb[1]
    w = max(tw + pad*2, 60)
    h2 = th + pad*2
    bg = make_bg(w, h2)
    draw = ImageDraw.Draw(bg)
    col = tuple(random.randint(0, 50) for _ in range(3))
    draw.text(((w-tw)//2, (h2-th)//2), display, font=font, fill=col)
    return bg

def augment(img):
    # Blur — the main robustness target (simulates camera blur)
    if random.random() < 0.65:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 3.5)))
    if random.random() < 0.5:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.55, 1.45))
    if random.random() < 0.45:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.6, 1.5))
    if random.random() < 0.35:
        arr = np.array(img).astype(np.float32)
        arr = np.clip(arr + np.random.normal(0, random.uniform(5,25), arr.shape), 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    if random.random() < 0.4:
        img = img.rotate(random.uniform(-5, 5), fillcolor=(255,255,255))
    if random.random() < 0.3:
        import io
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=random.randint(35, 70))
        buf.seek(0); img = Image.open(buf).copy()
    return img

# Test — should show properly connected Arabic letters
render("على ابراهيم", FONT_PATHS[0])

In [ ]:
# ── 5. Generate dataset ────────────────────────────────────────────────────
import csv
from pathlib import Path
from tqdm import tqdm

N       = 15000          # number of synthetic images — increase for better results
OUT     = Path('/content/arabic_synthetic')
(OUT / 'images').mkdir(parents=True, exist_ok=True)

rows = []
for i in tqdm(range(N)):
    text = sample_text()
    fp   = random.choice(FONT_PATHS)
    try:
        img = augment(render(text, fp))
    except Exception:
        continue
    fname = f'syn_{i:06d}.png'
    img.save(OUT / 'images' / fname)
    rows.append({'filename': fname, 'text': text})

with open(OUT / 'labels.csv', 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, ['filename','text']).writerows([{'filename':'filename','text':'text'}] + rows)

print(f'Generated {len(rows)} images')

In [ ]:
# ── 6. Processor + tokenizer setup ────────────────────────────────────────
# Encoder: google/vit-base-patch16-224  (pure ViT — language-agnostic image features)
# Decoder: aubmindlab/bert-base-arabertv2 (Arabic BERT — knows Arabic script)
# NOTE: microsoft/trocr-base-printed is itself an encoder-decoder, so it can't
#       be reused as just the encoder. We use a plain ViT instead.

import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
from transformers import ViTImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

ENCODER_MODEL = 'google/vit-base-patch16-224'
DECODER_MODEL = 'aubmindlab/bert-base-arabertv2'

print("Loading ViT image processor...")
feature_extractor = ViTImageProcessor.from_pretrained(ENCODER_MODEL)

print("Loading Arabic tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)

# Sanity check — Arabic must round-trip correctly
test_ids = tokenizer("على ابراهيم", return_tensors="pt").input_ids
decoded  = tokenizer.decode(test_ids[0], skip_special_tokens=True)
print(f"Tokenizer Arabic test: '{decoded}'")
assert "على" in decoded, "Arabic tokenizer not working!"

df = pd.read_csv(OUT / 'labels.csv')
print(f"\nDataset: {len(df)} images")


class ArabicOCRDataset(Dataset):
    def __init__(self, df, img_dir, max_len=64):
        self.df      = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.max_len = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['filename']).convert('RGB')

        pixel_values = feature_extractor(
            images=img, return_tensors='pt'
        ).pixel_values.squeeze(0)

        labels = tokenizer(
            str(row['text']),
            padding='max_length',
            max_length=self.max_len,
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)

        labels[labels == tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

print("Dataset class ready ✓")

In [ ]:
# ── 7. Train / Val / Test split ───────────────────────────────────────────
from sklearn.model_selection import train_test_split

# 80% train | 10% val | 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f'Train : {len(train_df)}')
print(f'Val   : {len(val_df)}')
print(f'Test  : {len(test_df)}')

train_ds = ArabicOCRDataset(train_df, OUT / 'images')
val_ds   = ArabicOCRDataset(val_df,   OUT / 'images')
test_ds  = ArabicOCRDataset(test_df,  OUT / 'images')

In [ ]:
# ── 8. Build model (ViT encoder + Arabic BERT decoder) ────────────────────
from transformers import (
    VisionEncoderDecoderModel,
    ViTModel,
    BertLMHeadModel,
    BertConfig,
)

print("Loading ViT encoder...")
encoder = ViTModel.from_pretrained(ENCODER_MODEL)

print("Loading Arabic BERT decoder...")
decoder_config = BertConfig.from_pretrained(DECODER_MODEL)
decoder_config.is_decoder = True           # enable cross-attention
decoder_config.add_cross_attention = True
decoder = BertLMHeadModel.from_pretrained(DECODER_MODEL, config=decoder_config)

print("Combining into VisionEncoderDecoder...")
model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder)

# Required config
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id           = tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size
model.config.eos_token_id           = tokenizer.sep_token_id
model.config.max_length             = 64
model.config.num_beams              = 4
model.config.early_stopping         = True
model.config.no_repeat_ngram_size   = 3

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model ready ✓  ({total:.0f}M parameters)")

In [ ]:
# ── 9. CER metric ─────────────────────────────────────────────────────────
import editdistance
import numpy as np

def compute_cer(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids.copy()

    # Replace -100 (ignored padding) back to pad token for decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    chars, errs = 0, 0
    for p, r in zip(pred_str, label_str):
        chars += max(len(r), 1)
        errs  += editdistance.eval(p, r)

    cer = round(errs / chars, 4)
    print(f"  Sample pred : {pred_str[0][:40]!r}")
    print(f"  Sample ref  : {label_str[0][:40]!r}")
    return {'cer': cer}

In [ ]:
# ── 10. Train ──────────────────────────────────────────────────────────────
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    predict_with_generate=True,
    eval_strategy='epoch',        # renamed from evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='cer',
    greater_is_better=False,
    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_steps=300,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    save_total_limit=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_cer,
)

trainer.train()

In [ ]:
# ── 11. Save + download ────────────────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)

# Save model info so we know how to load it locally
import json
info = {'encoder': ENCODER_MODEL, 'decoder': DECODER_MODEL}
with open(f'{OUTPUT_DIR}/model_info.json', 'w') as f:
    json.dump(info, f)

print(f'Saved to {OUTPUT_DIR}')

import shutil
shutil.make_archive('/content/trocr-arabic-id', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/trocr-arabic-id.zip')
print('Download started ✓')

In [ ]:
# ── 11b. Test set evaluation ───────────────────────────────────────────────
# Run on the held-out test set — these images were never seen during training
from torch.utils.data import DataLoader
from tqdm import tqdm

model.eval()
loader = DataLoader(test_ds, batch_size=16)

all_preds, all_refs = [], []
with torch.no_grad():
    for batch in tqdm(loader, desc='Testing'):
        pv        = batch['pixel_values'].to(model.device)
        label_ids = batch['labels']

        generated = model.generate(pv)
        preds = tokenizer.batch_decode(generated, skip_special_tokens=True)

        label_ids[label_ids == -100] = tokenizer.pad_token_id
        refs  = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        all_preds.extend(preds)
        all_refs.extend(refs)

# CER
import editdistance
chars = sum(max(len(r), 1) for r in all_refs)
errs  = sum(editdistance.eval(p, r) for p, r in zip(all_preds, all_refs))
test_cer = round(errs / chars, 4)

# Exact match accuracy
exact = sum(p == r for p, r in zip(all_preds, all_refs))
accuracy = round(exact / len(all_refs), 4)

print(f'\n── Test Results ──────────────────────')
print(f'  Samples : {len(all_refs)}')
print(f'  CER     : {test_cer}   (lower = better, 0.0 = perfect)')
print(f'  Accuracy: {accuracy:.1%}  (exact match)')
print(f'─────────────────────────────────────')

# Show 10 random samples
import random
print(f'\n{"Ground truth":<35}  {"Predicted":<35}  Match')
print('-' * 76)
sample_idx = random.sample(range(len(all_refs)), min(10, len(all_refs)))
for i in sample_idx:
    match = '✓' if all_preds[i] == all_refs[i] else '✗'
    print(f'{all_refs[i][:33]:<35}  {all_preds[i][:33]:<35}  {match}')

In [ ]:
# ── 12. Quick evaluation ───────────────────────────────────────────────────
import torch
model.eval()

print(f'{"Ground truth":<35}  {"Predicted":<35}')
print('-' * 72)
for _, row in val_df.sample(10, random_state=0).iterrows():
    img = Image.open(OUT / 'images' / row['filename']).convert('RGB')
    pv  = feature_extractor(images=img, return_tensors='pt').pixel_values
    with torch.no_grad():
        ids = model.generate(pv)
    pred = tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
    print(f'{str(row["text"])[:33]:<35}  {pred[:33]:<35}')